# Master

In [1]:
import sys; sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from pathlib import Path
import lightgbm as lgb
from src.data import load_master
from src.select import null_importance

INTERIM = Path("../data/interim")

In [2]:
X, y = load_master()
X.shape

(307511, 3028)

# Null importance

Target-permutation importance. Fit a GBM once on the real target for actual gain, then many times on a **shuffled** target for a null gain distribution per feature. Each feature is judged against **itself** under a broken target — same distribution, cardinality and correlations — so the noise floor is fair (a random-noise probe mis-sets it because its distribution doesn't match the real features). Keep a feature only if its real gain beats its **own** 95th-percentile null. Fast fit (subsample + fixed trees) keeps the shuffles cheap.

In [3]:
gbm = lambda: lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31,
                                 min_child_samples=50, subsample=0.8, colsample_bytree=0.7,
                                 n_jobs=-1, verbose=-1)
res = null_importance(gbm, X, y, k=20, n=80_000)
res.head(30)

null run  1/20
null run  2/20
null run  3/20
null run  4/20
null run  5/20
null run  6/20
null run  7/20
null run  8/20
null run  9/20
null run 10/20
null run 11/20
null run 12/20
null run 13/20
null run 14/20
null run 15/20
null run 16/20
null run 17/20
null run 18/20
null run 19/20
null run 20/20


,actual,null_mean,null_95,null_max,keep
ext_source_mean,21767.775468,268.621161,375.366689,429.073634,True
ext_source_median,11745.253679,290.909762,452.159394,487.648033,True
credit_to_goods,2879.287025,301.556277,417.415948,421.751139,True
ext_2_3,2306.729718,378.944405,441.194988,466.798450,True
credit_to_annuity,1718.768555,316.022613,399.169208,540.802921,True
AMT_ANNUITY,1706.324236,376.813113,471.637000,522.727288,True
ext_source_max,1647.955853,327.718446,451.346447,480.177149,True
days_employed_percentage,1493.460193,432.283310,547.133795,602.316329,True
ext_source_min,1444.469131,317.263090,394.355887,409.742573,True
prev_DAYS_LAST_DUE_1ST_VERSION_max,1349.901253,237.420178,355.543467,401.418990,True


# Keep

In [4]:
kept = res.index[res["keep"]].tolist()
print(len(kept), "kept /", len(res), "| real gain beats own 95th-pct null")
res[res["keep"]].head(30)

354 kept / 3028 | real gain beats own 95th-pct null


,actual,null_mean,null_95,null_max,keep
ext_source_mean,21767.775468,268.621161,375.366689,429.073634,True
ext_source_median,11745.253679,290.909762,452.159394,487.648033,True
credit_to_goods,2879.287025,301.556277,417.415948,421.751139,True
ext_2_3,2306.729718,378.944405,441.194988,466.798450,True
credit_to_annuity,1718.768555,316.022613,399.169208,540.802921,True
AMT_ANNUITY,1706.324236,376.813113,471.637000,522.727288,True
ext_source_max,1647.955853,327.718446,451.346447,480.177149,True
days_employed_percentage,1493.460193,432.283310,547.133795,602.316329,True
ext_source_min,1444.469131,317.263090,394.355887,409.742573,True
prev_DAYS_LAST_DUE_1ST_VERSION_max,1349.901253,237.420178,355.543467,401.418990,True


# Save

In [5]:
pd.Series(kept, name="feature").to_csv(INTERIM / "null_importance_features.csv", index=False)
len(kept)

354